# Davinci — Persian / Bilingual Dataset Preparation

A compact, reproducible Google Colab workflow for preparing Persian-first and Persian–English corpora for language-model research.

**Scope:** dataset discovery → metadata/license audit → streaming/sample download → Persian normalization → quality filters → exact deduplication → token statistics → export + manifest.

> This notebook prepares data; it does not train the 500M model or claim that 500K-context capability is achieved. Dataset schemas and licenses vary. Inspect the dataset card and select the text column/config explicitly before processing.

For reusable Python pipelines outside Colab, the repository also provides `src/davinci/data_audit.py` and its tests. This notebook remains self-contained for Colab use.

## 1. Runtime and reproducibility

Use a Colab GPU only if you later add model training. Dataset preparation normally works on CPU. Set a seed and keep raw data separate from processed outputs.

In [ ]:
!pip -q install "datasets>=3.0.0,<4" "huggingface_hub>=0.25.0,<1" "pandas>=2.2,<3" "tqdm>=4.66.0,<5" "sentencepiece>=0.2.0,<0.3"

In [ ]:
import os, re, json, csv, hashlib, random, unicodedata, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from datasets import load_dataset, get_dataset_config_names
from huggingface_hub import HfApi
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/Davinci/data_pipeline")
RAW = OUT / "raw"
PROC = OUT / "processed"
OUT.mkdir(parents=True, exist_ok=True)
RAW.mkdir(exist_ok=True)
PROC.mkdir(exist_ok=True)
print("Python:", platform.python_version())
print("Output:", OUT)

## 2. Dataset registry

These are discovery candidates, not an assertion that every row is suitable or every upstream license permits every use. Start with one dataset, inspect its card/configuration, and record the exact subset used. Avoid combining a merged corpus with its component datasets without deduplication.

In [ ]:
DATASETS = {
    # General Persian / multilingual pretraining candidates
    "culturax_fa": {"id": "uonlp/CulturaX", "purpose": "pretraining", "language": "multilingual; inspect/filter Persian"},
    "fineweb2": {"id": "HuggingFaceFW/fineweb-2", "purpose": "pretraining", "language": "multilingual; select Persian subset/config"},
    "mc4": {"id": "allenai/mc4", "purpose": "pretraining", "language": "multilingual; inspect Persian config"},
    "persian_corpus_merged": {"id": "mshojaei77/PersianCorpus_merged", "purpose": "pretraining", "language": "Persian"},
    "naab": {"id": "SLPL/naab", "purpose": "pretraining", "language": "Persian"},
    "persian_blog": {"id": "RohanAiLab/persian_blog", "purpose": "pretraining", "language": "Persian"},
    # Instruction / multilingual candidates
    "farsinstruct": {"id": "ParsiAI/FarsInstruct", "purpose": "instruction", "language": "Persian"},
    "persian_sft_102k": {"id": "amiralifirouzi/Persian-SFT-102K", "purpose": "instruction", "language": "Persian + English"},
    "aya": {"id": "CohereForAI/aya_dataset", "purpose": "instruction", "language": "multilingual"},
    "xp3": {"id": "bigscience/xP3", "purpose": "instruction", "language": "multilingual"},
    # Reasoning candidate; verify generated content carefully
    "persian_alpaca_reasoning": {"id": "hosseinhimself/persian-alpaca-reasoning-v1", "purpose": "reasoning", "language": "Persian"},
}

for key, item in DATASETS.items():
    print(f"{key:30} {item['id']:55} {item['purpose']}")

## 3. Inspect dataset card, license metadata, and available configurations

This step queries Hugging Face metadata. A missing license field is **unknown**, not permission. A dataset-level license may not cover all upstream components in an aggregate dataset. Open the returned card URL and review its terms before using or redistributing data.

In [ ]:
api = HfApi()

def inspect_dataset(dataset_id):
    info = api.dataset_info(dataset_id, files_metadata=False)
    card = info.card_data.to_dict() if info.card_data else {}
    try:
        configs = get_dataset_config_names(dataset_id)
    except Exception as exc:
        configs = []
        config_error = str(exc)
    else:
        config_error = None
    return {
        "id": dataset_id,
        "revision": getattr(info, "sha", None),
        "card_url": f"https://huggingface.co/datasets/{dataset_id}",
        "license_declared": card.get("license"),
        "language_metadata": card.get("language"),
        "configs": configs,
        "config_error": config_error,
        "last_modified": str(info.last_modified) if info.last_modified else None,
        "private": info.private,
    }

selected_key = "persian_corpus_merged"  # Change this after reviewing the registry.
selected = DATASETS[selected_key]
metadata = inspect_dataset(selected["id"])
print(json.dumps(metadata, ensure_ascii=False, indent=2))
if not metadata["revision"]:
    raise RuntimeError("The Hub did not return an immutable dataset revision; stop rather than stream a moving dataset reference.")

# Explicit human audit; do not change to True until you have reviewed the card and upstream terms.
LICENSE_REVIEWED = False
LICENSE_APPROVED_FOR_USE = False  # True only when the reviewed terms permit your intended use.
if LICENSE_APPROVED_FOR_USE and not LICENSE_REVIEWED:
    raise ValueError("Cannot approve a dataset before recording a license review.")
LICENSE_REVIEW_NOTES = "Review dataset card, upstream sources, commercial-use terms, redistribution restrictions, and privacy."

## 4. Choose configuration, split, and inspect the schema

Do not assume column names. The following cell lists available configs, splits, and columns. Set the values only after reviewing the output.

In [ ]:
# Set to an actual config printed above; use None only if the loader supports a default config.
CONFIG_NAME = None
SPLIT = "train"

load_kwargs = {"split": SPLIT, "streaming": True, "revision": metadata["revision"]}
if CONFIG_NAME is not None:
    load_kwargs["name"] = CONFIG_NAME

stream = load_dataset(selected["id"], **load_kwargs)
print("Features:", stream.features)
print("First example:")
first = next(iter(stream.take(1)))
print(json.dumps(first, ensure_ascii=False, indent=2, default=str)[:5000])
print("\nSet TEXT_COLUMN (and optionally ID_COLUMNS) after inspecting this example.")

## 5. Select the text field and stream a bounded sample

This notebook intentionally requires an explicit text column. That prevents silently concatenating unrelated fields such as labels, metadata, or answers. Set a conservative row cap for the first run.

In [ ]:
TEXT_COLUMN = None  # REQUIRED: e.g. "text", only after checking Features / example above
MAX_ROWS = 20_000    # Increase deliberately after a successful pilot

if not TEXT_COLUMN:
    raise ValueError("Set TEXT_COLUMN to a verified text field from the dataset schema.")

# Reload a fresh stream because the previous iterator was consumed.
stream = load_dataset(selected["id"], **load_kwargs)
rows = []
for i, row in enumerate(tqdm(stream, total=MAX_ROWS)):
    if i >= MAX_ROWS:
        break
    value = row.get(TEXT_COLUMN)
    if isinstance(value, str) and value.strip():
        rows.append({"source_id": selected["id"], "source_row": i, "text": value})

print("Collected rows:", len(rows))
if rows:
    print(rows[0]["text"][:1000])

## 6. Conservative Persian normalization

Normalize Arabic Yeh/Kaf variants and Unicode composition; preserve ZWNJ (نیم‌فاصله), punctuation, digits, and meaning-bearing symbols. This is deliberately conservative. Keep the original text alongside normalized text if you need full traceability.

In [ ]:
ARABIC_TO_PERSIAN = str.maketrans({"ي":"ی", "ى":"ی", "ك":"ک"})

def normalize_fa(text):
    text = unicodedata.normalize("NFC", text)
    text = text.translate(ARABIC_TO_PERSIAN)
    text = text.replace("\u0640", "")  # tatweel
    text = re.sub(r"[\u200e\u200f\u202a-\u202e]", "", text)  # directional controls
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for row in rows:
    row["text_original"] = row["text"]
    row["text"] = normalize_fa(row["text"])
print("Normalization complete.")

## 7. Quality filters and exact deduplication

These are transparent baseline heuristics, not a substitute for human review or language identification. Adjust thresholds after inspecting samples. Exact-hash deduplication catches byte-identical normalized strings; it does not detect near-duplicates or cross-dataset overlap.

In [ ]:
MIN_CHARS = 40
MAX_CHARS = 200_000

FA_RE = re.compile(r"[\u0600-\u06FF]")
LATIN_RE = re.compile(r"[A-Za-z]")

def quality_flags(text):
    chars = len(text)
    words = len(text.split())
    fa_chars = len(FA_RE.findall(text))
    alpha_chars = sum(ch.isalpha() for ch in text)
    replacement = text.count("\ufffd")
    return {
        "chars": chars,
        "words": words,
        "persian_char_ratio": fa_chars / max(alpha_chars, 1),
        "replacement_char_count": replacement,
        "too_short": chars < MIN_CHARS,
        "too_long": chars > MAX_CHARS,
        "low_alpha": alpha_chars < 20,
    }

seen = set()
clean_rows, rejected_rows = [], []
for row in tqdm(rows):
    text = row["text"]
    flags = quality_flags(text)
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    reason = None
    if flags["too_short"]: reason = "too_short"
    elif flags["too_long"]: reason = "too_long"
    elif flags["low_alpha"]: reason = "low_alpha"
    elif flags["replacement_char_count"] > 0: reason = "replacement_character"
    elif digest in seen: reason = "exact_duplicate"
    if reason:
        rejected_rows.append({"source_row": row["source_row"], "reason": reason, **flags})
        continue
    seen.add(digest)
    clean_rows.append({**row, "sha256": digest, **flags})

print({"input":len(rows), "kept":len(clean_rows), "rejected":len(rejected_rows)})
print(pd.Series([r["reason"] for r in rejected_rows]).value_counts(dropna=False))

## 8. Inspect quality and language distribution

A simple script-ratio estimate is not a language classifier. For bilingual data, keep Persian and English examples and label them in a separate, auditable stage rather than dropping English by default.

In [ ]:
if clean_rows:
    df = pd.DataFrame(clean_rows)
    display(df[["chars","words","persian_char_ratio","source_row"]].describe())
    display(df.head(5)[["source_row","chars","words","persian_char_ratio","text"]])
    print("Rows with visible Persian script:", int((df.persian_char_ratio > 0).sum()))
else:
    print("No rows passed. Review TEXT_COLUMN and filter thresholds.")

## 9. Optional tokenizer-based token audit

Use the tokenizer intended for the model once it exists. A pretrained tokenizer can provide a rough comparison, but its counts are not the final model's token counts. Do not use this step to claim 500K-context readiness.

In [ ]:
TOKENIZER_ID = None  # Optional HF tokenizer ID; set only if you have selected one

tokenizer = None
if TOKENIZER_ID:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)
    token_counts = [len(tokenizer.encode(r["text"], add_special_tokens=False)) for r in clean_rows]
    if token_counts:
        print(pd.Series(token_counts).describe(percentiles=[.5,.9,.95,.99]))
else:
    print("Skipped: set TOKENIZER_ID to audit with a specific tokenizer.")

## 10. Export processed data and audit manifest

Exports JSONL with provenance fields, a rejection report, and a manifest. The license status remains pending unless you explicitly reviewed it. Keep raw downloads and processed outputs versioned separately; do not publish raw corpora until licensing/privacy review is complete.

In [ ]:
stamp = datetime.now(timezone.utc).isoformat()
jsonl_path = PROC / f"{selected_key}.jsonl"
reject_path = PROC / f"{selected_key}_rejected.csv"
manifest_path = PROC / f"{selected_key}_manifest.json"

VALIDATION_FRACTION = 0.01
def stable_split(digest, seed=SEED, validation_fraction=VALIDATION_FRACTION):
    bucket = int(hashlib.sha256(f"{seed}:{digest}".encode("ascii")).hexdigest()[:16], 16)
    return "validation" if bucket < int(validation_fraction * (1 << 64)) else "train"

for row in clean_rows:
    row["split"] = stable_split(row["sha256"])

train_rows = [row for row in clean_rows if row["split"] == "train"]
validation_rows = [row for row in clean_rows if row["split"] == "validation"]
if not train_rows or not validation_rows:
    raise ValueError("The deterministic split produced an empty partition; collect more data or adjust VALIDATION_FRACTION.")

with jsonl_path.open("w", encoding="utf-8") as f:
    for r in clean_rows:
        f.write(json.dumps({k:r[k] for k in ["source_id","source_row","text","sha256","split"]}, ensure_ascii=False) + "\n")

train_path = PROC / f"{selected_key}_train.jsonl"
validation_path = PROC / f"{selected_key}_validation.jsonl"
for output_path, partition in ((train_path, train_rows), (validation_path, validation_rows)):
    with output_path.open("w", encoding="utf-8") as f:
        for r in partition:
            f.write(json.dumps({k:r[k] for k in ["source_id","source_row","text","sha256","split"]}, ensure_ascii=False) + "\n")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

pd.DataFrame(rejected_rows).to_csv(reject_path, index=False)
manifest = {
    "created_at_utc": stamp,
    "dataset_id": selected["id"],
    "dataset_revision": metadata["revision"],
    "dataset_card_url": metadata["card_url"],
    "purpose": selected["purpose"],
    "license_declared_by_hub": metadata["license_declared"],
    "license_reviewed_by_user": LICENSE_REVIEWED,
    "license_approved_for_use": LICENSE_APPROVED_FOR_USE,
    "license_review_notes": LICENSE_REVIEW_NOTES,
    "config": CONFIG_NAME,
    "split": SPLIT,
    "text_column": TEXT_COLUMN,
    "max_rows_requested": MAX_ROWS,
    "rows_collected": len(rows),
    "rows_kept": len(clean_rows),
    "rows_rejected": len(rejected_rows),
    "validation_fraction": VALIDATION_FRACTION,
    "train_rows": len(train_rows),
    "validation_rows": len(validation_rows),
    "combined_jsonl_sha256": sha256_file(jsonl_path),
    "train_jsonl_sha256": sha256_file(train_path),
    "validation_jsonl_sha256": sha256_file(validation_path),
    "split_method": "SHA-256(seed:normalized_text_sha256) threshold; deterministic document-level split",
    "normalization": "NFC; Arabic Yeh/Kaf mapping; remove tatweel and bidi controls; preserve ZWNJ",
    "deduplication": "SHA-256 of normalized text; exact duplicates only",
    "filters": {"min_chars":MIN_CHARS, "max_chars":MAX_CHARS, "low_alpha_min":20},
    "seed": SEED,
    "package_versions": {name: importlib.metadata.version(name) for name in ["datasets", "huggingface_hub", "pandas", "tqdm", "sentencepiece"]},
    "output": str(jsonl_path),
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", jsonl_path, train_path, validation_path, reject_path, manifest_path, sep="\n")

## 11. Download artifacts from Colab

Run this cell to download the processed JSONL and audit files. For larger datasets, prefer saving to mounted Google Drive rather than downloading through the browser.

In [ ]:
from google.colab import files
for path in [jsonl_path, reject_path, manifest_path]:
    files.download(str(path))

## 12. Next datasets and safe scaling

1. Repeat the inspection and sampling workflow for each source; do not assume schemas are shared.
2. Record dataset-level and upstream licenses separately, especially for merged datasets.
3. Deduplicate across sources using hashes, then add near-duplicate detection only after measuring its false-positive rate.
4. Add language identification and quality scoring with a manually reviewed validation sample.
5. Build separate manifests for pretraining, instruction, reasoning, and agent/tool trajectories.
6. For SFT, inspect conversation structure and map fields explicitly; do not flatten multi-turn examples blindly.
7. For long-context training, construct and evaluate long examples at increasing lengths (e.g. 8K → 32K → 64K → 128K → 256K → 500K). This notebook does not generate or validate such examples.

**Candidate source pages:**
- [CulturaX](https://huggingface.co/datasets/uonlp/CulturaX)
- [FineWeb2](https://huggingface.co/datasets/HuggingFaceFW/fineweb-2)
- [mC4](https://huggingface.co/datasets/allenai/mc4)
- [PersianCorpus_merged](https://huggingface.co/datasets/mshojaei77/PersianCorpus_merged)
- [Naab](https://huggingface.co/datasets/SLPL/naab)
- [FarsInstruct](https://huggingface.co/datasets/ParsiAI/FarsInstruct)
- [Persian-SFT-102K](https://huggingface.co/datasets/amiralifirouzi/Persian-SFT-102K)
- [Aya](https://huggingface.co/datasets/CohereForAI/aya_dataset)
- [xP3](https://huggingface.co/datasets/bigscience/xP3)

Always re-check dataset cards and upstream terms at the time of use.